In [ ]:
import gzip
import json
import pickle

import ipywidgets as widgets
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from IPython.display import VimeoVideo
from ipywidgets import interact
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from teaching_tools.widgets import ConfusionMatrixWidget


In [ ]:
VimeoVideo("696221191", h="275ffd1421", width=600)


In [ ]:
def wrangle(filename):
    with gzip.open(filename,"r") as f:
        data = json.load(f)

     # Load dictionary into DataFrame, set index
    df = pd.DataFrame().from_dict(data["data"]).set_index("company_id")
    return df


In [ ]:
df = wrangle("data/poland-bankruptcy-data-2009.json.gz")
print(df.shape)
df.head()


In [ ]:
target = "bankrupt"
X = df.drop(columns=target)
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


In [ ]:
over_sampler = RandomOverSampler(random_state=42)
X_train_over, y_train_over = over_sampler.fit_resample(X_train, y_train)
print("X_train_over shape:", X_train_over.shape)
X_train_over.head()


In [ ]:
acc_baseline = y_train.value_counts(normalize=True).max()
print("Baseline Accuracy:", round(acc_baseline, 4))


In [ ]:
VimeoVideo("696221115", h="44fe95d5d9", width=600)


In [ ]:
clf = make_pipeline(
    SimpleImputer(),
    GradientBoostingClassifier()
)


In [ ]:
VimeoVideo("696221055", h="b675d7fec0", width=600)


In [ ]:
params = {
    "simpleimputer__strategy": ["mean","median"],
    "gradientboostingclassifier__n_estimators" : range(20,31,5),
    "gradientboostingclassifier__max_depth": range(2,5)

}
params


In [ ]:
VimeoVideo("696221023", h="218915d38e", width=600)


In [ ]:
model = GridSearchCV(
    clf, param_grid=params, cv=5, n_jobs=-1, verbose=1
)


In [ ]:
VimeoVideo("696220978", h="008d915f33", width=600)


In [ ]:
# Fit model to over-sampled training data
model.fit(X_train_over,y_train_over)


In [ ]:
VimeoVideo("696220937", h="9148032400", width=600)


In [ ]:
results = pd.DataFrame(model.cv_results_)
results.sort_values("rank_test_score").head(10)


In [ ]:
VimeoVideo("696220899", h="342d55e7d7", width=600)


In [ ]:
# Extract best hyperparameters
model.best_params_


In [ ]:
acc_train = model.score(X_train, y_train)
acc_test = model.score(X_test,y_test)

print("Training Accuracy:", round(acc_train, 4))
print("Validation Accuracy:", round(acc_test, 4))


In [ ]:
# Plot confusion matrix
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)


In [ ]:
VimeoVideo("696297886", h="fac5454b22", width=600)


In [ ]:
# Print classification report
print(classification_report(y_test, model.predict(X_test)))


In [ ]:
VimeoVideo("696220837", h="f93be5aba0", width=600)


In [ ]:
VimeoVideo("696220785", h="8a4c4bff58", width=600)


In [ ]:
c = ConfusionMatrixWidget(model, X_test, y_test)
c.show()


In [ ]:
VimeoVideo("696209314", h="36a14b503c", width=600)


In [ ]:
c.show_eu()


In [ ]:
VimeoVideo("696209348", h="f7e1981c9f", width=600)


In [ ]:
def make_cnf_matrix(threshold):
   
    y_pred_proba = model.predict_proba(X_test)[:,-1]

    y_pred = y_pred_proba > threshold
    conf_matrix = confusion_matrix(y_test, y_pred)
    tn,fp,fn,tp = conf_matrix.ravel()
    print(f" profit ${tp*100_000_000}")
    print(f" loss ${fp*250_000_000}")
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, colorbar=False)


thresh_widget = widgets.FloatSlider(min=0, max=1, value=0.5, step=0.05  )

interact(make_cnf_matrix, threshold=thresh_widget);


In [ ]:
# Save model
with open("model-5-4.pkl", "wb") as f:
    pickle.dump(model,f)


In [ ]:
VimeoVideo("696220731", h="8086ff0bcd", width=600)


In [ ]:
%%bash

cat my_predictor_lesson.py


In [ ]:
VimeoVideo("696220643", h="8a3f141262", width=600)


In [ ]:
# Import your module
from my_predictor_lesson import make_predictions

# Generate predictions
y_test_pred = make_predictions(
    data_filepath="data/poland-bankruptcy-data-2009-mvp-features.json.gz",
    model_filepath="model-5-4.pkl",
)

print("predictions shape:", y_test_pred.shape)
y_test_pred.head()
